In [1]:
import sys
# Ensure we aren't accidentally pulling from Roaming
sys.path = [p for p in sys.path if "Roaming" not in p]

try:
    from scipy import sparse
    import sklearn
    print("Success! Scipy Sparse and Sklearn are both loaded.")
except ImportError as e:
    print(f"Still failing: {e}")

Success! Scipy Sparse and Sklearn are both loaded.


In [2]:
import unsloth
import json
from datasets import Dataset
import sys
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig, GRPOConfig, GRPOTrainer
import torch

import os
from dotenv import load_dotenv
load_dotenv()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


True

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
#from langchain_community.chains import LLMChain
import json
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
import os
import urllib.request
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings, OpenAIEmbeddings
#from langchain.evaluation import load_evaluator
from datasets import Dataset
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from datasets import load_dataset
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.base import Metric
from langchain_experimental.text_splitter import SemanticChunker

import random
from typing import List
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser
from prompts import  RFT_EVAL_DATA_GEN_CONCISE

from pydantic import BaseModel, Field
from langchain_community.vectorstores import FAISS
#import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from IPython.display import display, Markdown
from transformers import TrainingArguments

import gc
import re
load_dotenv()

/tmp/ipykernel_3162/3144420548.py:32: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
/tmp/ipykernel_3162/3144420548.py:32: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
/tmp/ipykernel_3162/3144420548.py:32: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrect

True

In [4]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


In [5]:
import torch
print(f"PyTorch CUDA: {torch.version.cuda}")
print(torch.cuda.get_device_capability())
!nvcc --version

PyTorch CUDA: 12.8
(7, 5)


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [6]:
!nvidia-smi

Sun Mar  8 17:03:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   34C    P0             31W /   70W |     301MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Loading the dataset:

In [5]:
gsm8k_dataset = load_dataset('openai/gsm8k', 'socratic')
gsm8k_dataset_train = gsm8k_dataset['train']
gsm8k_dataset_train

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [6]:
gsm8k_dataset = load_dataset('openai/gsm8k', 'socratic')
gsm8k_dataset_test = gsm8k_dataset['test']
gsm8k_dataset_test

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})

Let's check out the dataset:

In [7]:
print("Checking the Question :")
display(Markdown(gsm8k_dataset_train[:1]['question'][0]))
print("Checking the Answer :")
display(Markdown(gsm8k_dataset_train[:1]['answer'][0]))

Checking the Question :


Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Checking the Answer :


How many clips did Natalia sell in May? ** Natalia sold 48/2 = <<48/2=24>>24 clips in May.
How many clips did Natalia sell altogether in April and May? ** Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

In [8]:
print("Checking the Question :")
display(Markdown(gsm8k_dataset_test[:1]['question'][0]))
print("Checking the Answer :")
display(Markdown(gsm8k_dataset_test[:1]['answer'][0]))

Checking the Question :


Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Checking the Answer :


How many eggs does Janet sell? ** Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
How much does Janet make at the farmers' market? ** She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

Get the base model to be fine-tuned:

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = 2048, 
    load_in_4bit = True,  
    load_in_8bit = False, 
    full_finetuning = False, 
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


Converting the base model to PEFT enabled model

In [8]:
model = FastLanguageModel.get_peft_model(
    model=model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)
print("PEFT enabled LLAMA3.1:8b model ready..!")

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PEFT enabled LLAMA3.1:8b model ready..!


Formatting train data as suitable for LLAMA:

In [9]:
SYSTEM_PROMPT_SFT = """
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Input:
{}

### Response:
{}
"""

In [10]:
def llama_formatting(dataset):
    formatted_data = []
    for question, answer in zip(dataset['question'], dataset['answer']):
        text = SYSTEM_PROMPT_SFT.format(question, answer) + tokenizer.eos_token
        formatted_data.append(text)
    return {'text':formatted_data}
    

In [11]:
train_dataset = gsm8k_dataset_train.map(llama_formatting,batched=True)
print(train_dataset)
print("Train Dataset : ",train_dataset[90])
print("----------------------------------------------------------------------------")
test_dataset = gsm8k_dataset_test.map(llama_formatting,batched=True)
print(test_dataset)
print("Test Dataset : ",test_dataset[90])

Dataset({
    features: ['question', 'answer', 'text'],
    num_rows: 7473
})
Train Dataset :  {'question': 'Herman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?', 'answer': 'How many days are in the three months? ** December has 31 days, January has 31 days and February has 28 days for a total of 31+31+28 = <<31+31+28=90>>90 days\nHow many cups does he feed them in total? ** He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon for a total of 1/2+1/2 = <<1/2+1/2=1>>1 cup per day\nHow many cups of birdseed will he need? ** If he feeds them 1 cup per day for 90 days then he will need 1*90 = <<1*90=90>>90 cups of birdseed\n#### 90', 'text': '\nBelow is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Input:\nHerman likes to feed the b

In [12]:
display(Markdown(train_dataset[90]['text']))


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Input:
Herman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?

### Response:
How many days are in the three months? ** December has 31 days, January has 31 days and February has 28 days for a total of 31+31+28 = <<31+31+28=90>>90 days
How many cups does he feed them in total? ** He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon for a total of 1/2+1/2 = <<1/2+1/2=1>>1 cup per day
How many cups of birdseed will he need? ** If he feeds them 1 cup per day for 90 days then he will need 1*90 = <<1*90=90>>90 cups of birdseed
#### 90
<|end_of_text|>

Fine-Tuning the model:

In [13]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="text",

    args= TrainingArguments(
                output_dir="llama31-8bn_SFT", #This will also be used as your huggingfacehub model id name
                report_to="wandb", 
                run_name="SFT_Llama3.1-8bn_v1",
                warmup_steps = 5,
                eval_steps=5,
                eval_strategy="steps",
                per_device_train_batch_size=2,    # small batches if quantized
                per_device_eval_batch_size=1,
                gradient_accumulation_steps=8,
                learning_rate = 2e-4,
                #num_train_epochs=1,
                max_steps=70,                    # or set num_train_epochs
                save_strategy="no",
                weight_decay = 0.01,
                bf16=False,
                fp16=False,
                gradient_checkpointing=True,
                logging_strategy="steps",
                logging_steps=5,
                seed=42,
                optim="adamw_torch",
                lr_scheduler_type="cosine",
            )
    )


In [16]:
#Taken from https://medium.com/mitb-for-all/how-to-train-your-llm-teaching-toothless-to-bite-8d9f56fe4b2a
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
9.404 GB of memory reserved.


In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 70
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /teamspace/studios/this_studio/.netrc.
wandb: Currently logged in as: namrata-thakur5790 (namrata-thakur5790-ibm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, instructor, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
5,1.511826,1.098002
10,0.901263,0.783086
15,0.808129,0.749148
20,0.738635,0.737182
25,0.729466,0.728109
30,0.740399,0.720124
35,0.715198,0.716619
40,0.708780,0.711748
45,0.741065,0.709288
50,0.713636,0.707302


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 377, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during 

eval/loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁
eval/runtime,▂▁▂▃▄▄▅▆▆▇▇▇██
eval/samples_per_second,▇█▇▆▅▅▄▃▃▂▂▂▁▁
eval/steps_per_second,▇█▇▆▅▅▄▃▃▂▂▂▁▁
train/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/grad_norm,█▄▃▂▂▂▂▁▂▂▁▁▁▁
train/learning_rate,▇██▇▇▆▅▄▃▃▂▁▁▁
train/loss,█▃▂▂▂▂▁▁▂▁▂▂▁▁
eval/loss,0.70295
eval/runtime,463.0147


In [18]:
#!rm -rf ~/Fine-tuning-LLMs-Strategies/notebooks/unsloth_compiled_cache

In [17]:
#Taken from https://medium.com/mitb-for-all/how-to-train-your-llm-teaching-toothless-to-bite-8d9f56fe4b2a
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

7397.9988 seconds used for training.
123.3 minutes used for training.
Peak reserved memory = 9.404 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 63.795 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [18]:
#This is merge LoRA adapters with model weights before pushing to Huggingface Hub:
model.push_to_hub_merged("NamrataThakur/llama31-8bn_SFT", tokenizer, 
                         save_method = "merged_16bit", token = os.environ["HF_TOKEN"])

config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:47<01:35, 95.53s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:09<00:00, 77.36s/it]


Unsloth: Merge process complete. Saved to `/teamspace/studios/this_studio/Fine-tuning-LLMs-Strategies/notebooks/NamrataThakur/llama31-8bn_SFT`


In [19]:
#This is merge LoRA adapters with model weights before saving in the local directory:
model.save_pretrained_merged("llama31-8bn_SFT", tokenizer, save_method = "merged_16bit")

Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:51<00:00, 12.82s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:19<00:00, 19.99s/it]


Unsloth: Merge process complete. Saved to `/teamspace/studios/this_studio/Fine-tuning-LLMs-Strategies/notebooks/llama31-8bn_SFT`


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py ./llama31-8bn_SFT --outfile llama31-8bn_SFT_f16.gguf --outtype f16